In [0]:

# Phase 3: Silver Layer - Read Bronze Data

bronze_table = "workspace.customer_data_pipeline.bronze_customers"

df_bronze = spark.table(bronze_table)

display(df_bronze)


In [0]:
# Step 3: Check data quality


In [0]:

from pyspark.sql import functions as F

# Count records and missing values in Bronze
df_bronze.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("email").isNull().cast("int")).alias("missing_emails"),
    F.sum(F.col("city").isNull().cast("int")).alias("missing_cities"),
    (F.count("*") - F.countDistinct("customer_id")).alias("duplicate_customer_ids")
).show()


In [0]:

# Phase 3 - Step 4: Remove duplicate customers

df_silver = df_bronze.dropDuplicates(["customer_id"])

print("Bronze record count:", df_bronze.count())
print("Silver record count:", df_silver.count())

display(df_silver.orderBy("customer_id"))


In [0]:

# Phase 3 - Step 5: Handle missing values

df_silver = df_silver.fillna({
    "email": "Unknown",
    "city": "Unknown"
})

display(df_silver.orderBy("customer_id"))


In [0]:
#  phase -3: step 6: PHASE 8 - STEP 10B
# Validate cleaned incoming customer records.
#
# Purpose:
# Allow different batch sizes while checking
# for duplicate IDs and missing required values.

total_records = df_silver.count()

duplicate_ids = (
    total_records
    - df_silver.select("customer_id").distinct().count()
)

null_customer_ids = df_silver.filter(
    F.col("customer_id").isNull()
).count()

missing_emails = df_silver.filter(
    F.col("email").isNull()
).count()

missing_cities = df_silver.filter(
    F.col("city").isNull()
).count()

print("Silver batch records:", total_records)
print("Duplicate customer IDs:", duplicate_ids)
print("NULL customer IDs:", null_customer_ids)
print("Missing emails:", missing_emails)
print("Missing cities:", missing_cities)

assert duplicate_ids == 0, "Duplicate customer IDs found"
assert null_customer_ids == 0, "NULL customer IDs found"
assert missing_emails == 0, "Missing emails found"
assert missing_cities == 0, "Missing cities found"

print("SILVER BATCH DATA QUALITY VALIDATION PASSED")

In [0]:

# Phase 3 - Step 7: Write Silver Delta Table
# Phase 3 - Step 7: Write Silver Delta Table

# PHASE 8 - MIGRATION NOTE
#
# This is the original full-refresh Silver write.
# It must not run during incremental MERGE processing.
#
# We will replace this logic with Delta Lake MERGE
# after validating the new incremental path.

silver_table = "workspace.customer_data_pipeline.silver_customers"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver Delta table created successfully!")
print("Table:", silver_table)


In [0]:

# Phase 3 - Step 8: Verify Saved Silver Delta Table

df_verify = spark.table(
    "workspace.customer_data_pipeline.silver_customers"
)

saved_count = df_verify.count()

print("Saved Silver records:", saved_count)

assert saved_count == 4, "Silver table validation failed!"

print("SILVER DELTA TABLE VALIDATION PASSED")

display(df_verify.orderBy("customer_id"))


In [0]:
%sql
CREATE TABLE IF NOT EXISTS
workspace.customer_data_pipeline.silver_customers_phase8_backup
DEEP CLONE
workspace.customer_data_pipeline.silver_customers;

In [0]:
%sql
-- PHASE 8 - STEP 2: VERIFY SILVER BACKUP
--
-- Purpose:
-- Confirm the backup contains the same number of
-- customer records as the original Silver table.
--
-- Expected:
-- Original Silver = 4
-- Backup Silver   = 4

SELECT
    'Original Silver' AS table_name,
    COUNT(*) AS record_count
FROM workspace.customer_data_pipeline.silver_customers

UNION ALL

SELECT
    'Backup Silver' AS table_name,
    COUNT(*) AS record_count
FROM workspace.customer_data_pipeline.silver_customers_phase8_backup;

In [0]:
%sql
-- PHASE 8 - STEP 3: VERIFY BACKUP DATA
--
-- Purpose:
-- Check whether the original Silver table and
-- the backup contain exactly the same records.
--
-- Expected result:
-- 0 rows returned.

(
    SELECT *
    FROM workspace.customer_data_pipeline.silver_customers

    EXCEPT ALL

    SELECT *
    FROM workspace.customer_data_pipeline.silver_customers_phase8_backup
)

UNION ALL

(
    SELECT *
    FROM workspace.customer_data_pipeline.silver_customers_phase8_backup

    EXCEPT ALL

    SELECT *
    FROM workspace.customer_data_pipeline.silver_customers
);

In [0]:
%sql
-- PHASE 8 - STEP 4: VIEW EXISTING SILVER CUSTOMERS
--
-- Purpose:
-- Understand the existing customer records before
-- implementing incremental inserts and updates.
--
-- Expected:
-- 4 unique customers.

SELECT
    customer_id,
    customer_name,
    email,
    city,
    signup_date
FROM workspace.customer_data_pipeline.silver_customers
ORDER BY customer_id;

In [0]:
%sql
-- PHASE 8 - STEP 5A: CREATE INCREMENTAL CUSTOMER DATA
--
-- Purpose:
-- Simulate incoming customer changes.
--
-- 102 and 103: Existing customers with updated information.
-- 105 and 106: New customers.
--
-- This is a separate staging table.
-- The Silver table is not modified in this step.

CREATE OR REPLACE TABLE
workspace.customer_data_pipeline.incremental_customers
USING DELTA
AS
SELECT
    102 AS customer_id,
    'Bob Smith' AS customer_name,
    'bob.smith@example.com' AS email,
    'Austin' AS city,
    DATE '2026-01-08' AS signup_date

UNION ALL

SELECT
    103,
    'Carol Davis',
    'carol.davis@example.com',
    'San Antonio',
    DATE '2026-01-10'

UNION ALL

SELECT
    105,
    'Emma Brown',
    'emma.brown@example.com',
    'Dallas',
    DATE '2026-02-01'

UNION ALL

SELECT
    106,
    'Frank Miller',
    'frank.miller@example.com',
    'Houston',
    DATE '2026-02-02';

In [0]:
%sql
-- PHASE 8 - STEP 5B: VERIFY INCREMENTAL DATA
--
-- Purpose:
-- Confirm the incoming customer records are correct.
--
-- Expected:
-- 4 records total
-- 2 existing customers (102, 103)
-- 2 new customers (105, 106)

SELECT
    customer_id,
    customer_name,
    email,
    city,
    signup_date
FROM workspace.customer_data_pipeline.incremental_customers
ORDER BY customer_id;

In [0]:
%sql
-- PHASE 8 - STEP 6: VALIDATE INCOMING CUSTOMER IDS
--
-- Purpose:
-- Check for duplicate or NULL customer IDs
-- before performing Delta Lake MERGE.
--
-- Expected:
-- total_records = 4
-- unique_customer_ids = 4
-- null_customer_ids = 0
-- duplicate_customer_ids = 0

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS unique_customer_ids,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS null_customer_ids,
    COUNT(customer_id) - COUNT(DISTINCT customer_id)
        AS duplicate_customer_ids
FROM workspace.customer_data_pipeline.incremental_customers;

In [0]:
%sql
-- PHASE 8 - STEP 7A: INCREMENTAL CUSTOMER MERGE
--
-- Purpose:
-- Update existing customers and insert new customers.
--
-- Matching key: customer_id
--
-- Expected:
-- 2 updates (102, 103)
-- 2 inserts (105, 106)
-- Silver total after MERGE = 6

MERGE INTO workspace.customer_data_pipeline.silver_customers AS target
USING workspace.customer_data_pipeline.incremental_customers AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND NOT (
    target.customer_name <=> source.customer_name
    AND target.email <=> source.email
    AND target.city <=> source.city
    AND target.signup_date <=> source.signup_date
)
THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.email = source.email,
    target.city = source.city,
    target.signup_date = source.signup_date

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    email,
    city,
    signup_date
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.email,
    source.city,
    source.signup_date
);

In [0]:
%sql
-- PHASE 8 - STEP 7B: VERIFY MERGE RESULTS
--
-- Purpose:
-- Confirm existing customers were updated and
-- new customers were inserted successfully.
--
-- Expected:
-- 6 customer records after MERGE.

SELECT
    customer_id,
    customer_name,
    email,
    city,
    signup_date
FROM workspace.customer_data_pipeline.silver_customers
ORDER BY customer_id;

In [0]:
%sql
-- PHASE 8 - STEP 8B: VERIFY MERGE IDEMPOTENCY
--
-- Purpose:
-- Confirm that rerunning MERGE did not create
-- duplicate customer records.
--
-- Expected:
-- total_customers = 6
-- unique_customers = 6
-- duplicate_customers = 0

SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(DISTINCT customer_id)
        AS duplicate_customers
FROM workspace.customer_data_pipeline.silver_customers;

In [0]:
%sql
-- PHASE 8 - STEP 9A
-- Create a one-time backup before changing
-- the Silver pipeline from overwrite to MERGE.
--
-- Expected: Backup contains 6 customers.

CREATE TABLE IF NOT EXISTS
workspace.customer_data_pipeline.silver_customers_pre_incremental_backup
DEEP CLONE
workspace.customer_data_pipeline.silver_customers;

In [0]:
%sql
-- PHASE 8 - STEP 9B: VERIFY SILVER BACKUP
--
-- Purpose:
-- Compare the current Silver table with
-- the recovery backup created in Step 9A.
--
-- Expected:
-- Current Silver = 6
-- Backup Silver = 6

SELECT
    'Current Silver' AS table_name,
    COUNT(*) AS total_customers
FROM workspace.customer_data_pipeline.silver_customers

UNION ALL

SELECT
    'Backup Silver' AS table_name,
    COUNT(*) AS total_customers
FROM workspace.customer_data_pipeline.silver_customers_pre_incremental_backup;

In [0]:
# PHASE 8 - STEP 10C
# Inspect the Bronze table schema.
#
# Purpose:
# Identify available columns for deterministic
# deduplication and incremental processing.

bronze_table = "workspace.customer_data_pipeline.bronze_customers"

df_bronze_schema = spark.table(bronze_table)

df_bronze_schema.printSchema()

In [0]:
%sql
-- PHASE 8 - STEP 10D
-- Inspect duplicate customer IDs in Bronze.
--
-- Purpose:
-- Understand which customer records have duplicates
-- before designing deterministic deduplication.

SELECT
    customer_id,
    customer_name,
    email,
    city,
    signup_date
FROM workspace.customer_data_pipeline.bronze_customers
WHERE customer_id IN (
    SELECT customer_id
    FROM workspace.customer_data_pipeline.bronze_customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
)
ORDER BY customer_id;

In [0]:
# PHASE 8 - STEP 10E
# Remove exact duplicate records from Bronze.
#
# Purpose:
# Understand how Spark distinct() works.
#
# This does NOT modify the Bronze or Silver tables.

bronze_df = spark.table(
    "workspace.customer_data_pipeline.bronze_customers"
)

# Count original Bronze records
original_count = bronze_df.count()

# Remove rows where every column is identical
unique_df = bronze_df.distinct()

# Count records after removing exact duplicates
unique_count = unique_df.count()

print("Original Bronze records:", original_count)
print("After removing exact duplicates:", unique_count)
print("Exact duplicates removed:", original_count - unique_count)

In [0]:
# PHASE 8 - STEP 10F
# Prepare a clean source DataFrame for Delta MERGE.
#
# This step does NOT modify any Delta tables.

from pyspark.sql import functions as F

# Step 1: Read Bronze customer data
df_bronze = spark.table(
    "workspace.customer_data_pipeline.bronze_customers"
)

# Step 2: Remove completely identical records
df_unique = df_bronze.distinct()

# Step 3: Replace missing email and city values
df_merge_source = df_unique.fillna({
    "email": "Unknown",
    "city": "Unknown"
})

# Step 4: Validate the MERGE source
total_records = df_merge_source.count()

null_ids = df_merge_source.filter(
    F.col("customer_id").isNull()
).count()

duplicate_ids = (
    total_records
    - df_merge_source.select("customer_id").distinct().count()
)

print("MERGE source records:", total_records)
print("NULL customer IDs:", null_ids)
print("Duplicate customer IDs:", duplicate_ids)

assert null_ids == 0, "NULL customer IDs found"
assert duplicate_ids == 0, "Duplicate customer IDs found"

print("MERGE SOURCE VALIDATION PASSED")

In [0]:
%sql
-- PHASE 8 - STEP 10G
-- Create a separate target table for testing
-- the PySpark Delta Lake MERGE implementation.
--
-- Purpose:
-- Protect the real Silver table while developing.
--
-- Expected:
-- Test table contains 6 customers.

CREATE OR REPLACE TABLE
workspace.customer_data_pipeline.silver_customers_merge_test
DEEP CLONE
workspace.customer_data_pipeline.silver_customers;

In [0]:
%sql
-- PHASE 8 - STEP 10H
-- Verify our PySpark MERGE test table.
--
-- Purpose:
-- Confirm the test table has 6 unique customers.
--
-- Expected:
-- total_customers = 6
-- unique_customers = 6
-- duplicate_customers = 0

SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(DISTINCT customer_id)
        AS duplicate_customers
FROM workspace.customer_data_pipeline.silver_customers_merge_test;

In [0]:
# PHASE 8 - STEP 10I
# Test incremental MERGE using PySpark and Delta Lake.
#
# Source: incremental_customers (4 records)
# Target: silver_customers_merge_test (6 records)
#
# Expected:
# Existing customers remain updated.
# No new duplicate customers.
# Total target records remain 6.

from delta.tables import DeltaTable

# Step 1: Read incoming incremental customer data
df_incremental = spark.table(
    "workspace.customer_data_pipeline.incremental_customers"
)

# Step 2: Connect to our test Delta table
target_table = DeltaTable.forName(
    spark,
    "workspace.customer_data_pipeline.silver_customers_merge_test"
)

# Step 3: MERGE incoming records into the target
(
    target_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        condition="""
            NOT (
                target.customer_name <=> source.customer_name
                AND target.email <=> source.email
                AND target.city <=> source.city
                AND target.signup_date <=> source.signup_date
            )
        """,
        set={
            "customer_name": "source.customer_name",
            "email": "source.email",
            "city": "source.city",
            "signup_date": "source.signup_date"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "email": "source.email",
            "city": "source.city",
            "signup_date": "source.signup_date"
        }
    )
    .execute()
)

print("PYSPARK DELTA MERGE COMPLETED SUCCESSFULLY")

In [0]:
# PHASE 8 - STEP 10J
# Verify PySpark Delta MERGE results.
#
# Purpose:
# Check the latest MERGE operation metrics
# and confirm the target has no duplicate IDs.

from pyspark.sql import functions as F

test_table = (
    "workspace.customer_data_pipeline."
    "silver_customers_merge_test"
)

# Step 1: Read Delta Lake transaction history
history_df = spark.sql(f"""
    DESCRIBE HISTORY {test_table}
""")

# Step 2: Display the latest operation
history_df.select(
    "version",
    "operation",
    "operationMetrics"
).orderBy(
    F.col("version").desc()
).show(1, truncate=False)

# Step 3: Validate the target table
df_test = spark.table(test_table)

total_customers = df_test.count()

unique_customers = (
    df_test.select("customer_id")
    .distinct()
    .count()
)

duplicate_customers = (
    total_customers - unique_customers
)

print("Total customers:", total_customers)
print("Unique customers:", unique_customers)
print("Duplicate customers:", duplicate_customers)

assert total_customers == 6
assert duplicate_customers == 0

print("PYSPARK MERGE VALIDATION PASSED")

In [0]:
# PHASE 8 - STEP 10K
# Prepare incremental source for the Silver pipeline.
#
# Purpose:
# Read incoming changes from our staging table.
# Do not modify Bronze or Silver yet.

from pyspark.sql import functions as F

incremental_table = (
    "workspace.customer_data_pipeline.incremental_customers"
)

# Read the incoming customer changes
df_incoming = spark.table(incremental_table)

# Remove exact duplicate records
df_incoming = df_incoming.distinct()

# Clean missing values
df_incoming = df_incoming.fillna({
    "email": "Unknown",
    "city": "Unknown"
})

# Validate incoming customer IDs
incoming_count = df_incoming.count()

null_ids = df_incoming.filter(
    F.col("customer_id").isNull()
).count()

duplicate_ids = (
    incoming_count
    - df_incoming.select("customer_id").distinct().count()
)

assert null_ids == 0, "NULL customer IDs found"
assert duplicate_ids == 0, "Duplicate customer IDs found"

print("Incoming customer records:", incoming_count)
print("NULL customer IDs:", null_ids)
print("Duplicate customer IDs:", duplicate_ids)
print("INCREMENTAL SOURCE READY")

In [0]:
# PHASE 8 - STEP 10L
# MERGE cleaned incoming customers into Silver test table.
#
# Source: df_incoming from Step 10K
# Target: silver_customers_merge_test
#
# Purpose:
# Update existing customers when values change.
# Insert customers that do not exist.
# Avoid unnecessary updates on repeated runs.

from delta.tables import DeltaTable

# Connect to the Silver test table
silver_test = DeltaTable.forName(
    spark,
    "workspace.customer_data_pipeline.silver_customers_merge_test"
)

# Execute incremental MERGE
(
    silver_test.alias("target")
    .merge(
        df_incoming.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        condition="""
            NOT (
                target.customer_name <=> source.customer_name
                AND target.email <=> source.email
                AND target.city <=> source.city
                AND target.signup_date <=> source.signup_date
            )
        """,
        set={
            "customer_name": "source.customer_name",
            "email": "source.email",
            "city": "source.city",
            "signup_date": "source.signup_date"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "email": "source.email",
            "city": "source.city",
            "signup_date": "source.signup_date"
        }
    )
    .execute()
)

print("CLEANED INCREMENTAL MERGE COMPLETED")

In [0]:
%sql
-- PHASE 8 - STEP 10M
-- Verify the latest Delta Lake MERGE operation.
--
-- Purpose:
-- Check how many records were inserted,
-- updated, or deleted during Step 10L.
--
-- Expected:
-- Updated = 0
-- Inserted = 0
-- Deleted = 0

SELECT
    version,
    operation,
    operationMetrics['numSourceRows'] AS source_rows,
    operationMetrics['numTargetRowsUpdated'] AS updated_rows,
    operationMetrics['numTargetRowsInserted'] AS inserted_rows,
    operationMetrics['numTargetRowsDeleted'] AS deleted_rows
FROM (
    DESCRIBE HISTORY
    workspace.customer_data_pipeline.silver_customers_merge_test
)
ORDER BY version DESC
LIMIT 1;